In [233]:
from Bio import SeqIO
import gzip
import re
import pandas as pd

# -------- Parameters --------
motif = "CGAACNNNNGTTCG"
max_mismatches = 1
upstream_start = -370
upstream_end = -10

In [234]:
# -------- Helper Functions --------
def hamming_distance(s1, s2):
    return sum(a != b for a, b in zip(s1, s2))

def motif_match_with_N(motif, seq_window, max_mismatches=0):
    mismatches = 0
    for m, s in zip(motif, seq_window):
        if m == "N":
            continue  # N matches anything
        if m != s:
            mismatches += 1
            if mismatches > max_mismatches:
                return False, mismatches
    return True, mismatches


def reverse_complement(seq):
    complement = str.maketrans("ATCGN", "TAGCN")
    return seq.translate(complement)[::-1]

# -------- Load Genome --------
def load_genome(fasta_path):
    genome = {}
    with gzip.open(fasta_path, "rt") if fasta_path.endswith(".gz") else open(fasta_path) as handle:
        for record in SeqIO.parse(handle, "fasta"):
            genome[record.id] = str(record.seq).upper()
    return genome

# -------- Parse GTF (get TSS) --------
def parse_gtf_tss(gtf_path):
    tss_list = []
    with gzip.open(gtf_path, "rt") if gtf_path.endswith(".gz") else open(gtf_path) as f:
        for line in f:
            if line.startswith("#"): continue
            parts = line.strip().split("\t")
            if len(parts) < 9: continue
            if parts[2] != "gene": continue  # Only keep "gene" features

            chrom = parts[0]
            start = int(parts[3])
            end = int(parts[4])
            strand = parts[6]
            tss = start if strand == "+" else end

            # Parse attributes for GFF3: key=value;key2=value2
            attr_field = parts[8]
            attr_dict = dict(
                item.split("=", 1) for item in attr_field.strip().split(";") if "=" in item
            )

            # Try to get the gene name from 'Name' or 'gene' field
            gene_id = attr_dict.get("Name") or attr_dict.get("gene") or "unknown"

            tss_list.append((chrom, tss, strand, gene_id))
    return tss_list
# -------- Scan for Motif --------
def find_motifs_near_tss(genome, tss_list, motif, max_mismatches, upstream_start, upstream_end):
    motif_hits = []
    motif_len = len(motif)

    for chrom, tss, strand, gene_id in tss_list:
        if chrom not in genome:
            continue
        seq = genome[chrom]
        if strand == "+":
            region_start = max(0, tss + upstream_start)
            region_end = max(0, tss + upstream_end)
            region = seq[region_start:region_end]
        else:
            region_start = max(0, tss - upstream_end)
            region_end = max(0, tss - upstream_start)
            region = reverse_complement(seq[region_start:region_end])
        
        for i in range(0, len(region) - motif_len + 1):
            window = region[i:i+motif_len]
            matched, mismatches = motif_match_with_N(motif, window, max_mismatches)
            if matched:
                hit_position = tss + (i + upstream_start if strand == "+" else -i - upstream_start)
                # Compute absolute motif position first
                motif_start_abs = tss + (i + upstream_start if strand == "+" else -i - upstream_start)

                # Then compute relative position to TSS
                relative_pos = motif_start_abs - tss
                motif_hits.append({
                    "gene_id": gene_id,
                    "chrom": chrom,
                    "tss": tss,
                    "strand": strand,
                    "motif_start": hit_position,
                    "rel_position_motif": relative_pos,
                    "mismatches": mismatches,
                    "matched_seq": window
                })

    return motif_hits


In [235]:
# -------- Main Runner --------
def main():
    genome_fasta = "AM1001-oriented.fasta"       # Replace with your genome file
    gtf_file = "annot.gff"     # Replace with your annotation file

    genome = load_genome(genome_fasta)
    tss_list = parse_gtf_tss(gtf_file)
    hits = find_motifs_near_tss(genome, tss_list, motif, max_mismatches, upstream_start, upstream_end)

    for hit in hits:
        print(hit)

    df = pd.DataFrame(hits)
    df.to_csv("motif_hits.csv", index=False)
    print("Saved results to motif_hits.csv")

if __name__ == "__main__":
    main()

{'gene_id': 'pabC', 'chrom': 'tig00000001_polypolish', 'tss': 85555, 'strand': '+', 'motif_start': 85224, 'rel_position_motif': -331, 'mismatches': 1, 'matched_seq': 'CGCACAAGTGTTCG'}
{'gene_id': 'cwlD', 'chrom': 'tig00000001_polypolish', 'tss': 157408, 'strand': '+', 'motif_start': 157153, 'rel_position_motif': -255, 'mismatches': 1, 'matched_seq': 'CAAACGCGCGTTCG'}
{'gene_id': 'pgaptmp_000328', 'chrom': 'tig00000001_polypolish', 'tss': 302003, 'strand': '+', 'motif_start': 301693, 'rel_position_motif': -310, 'mismatches': 1, 'matched_seq': 'CGTACCACAGTTCG'}
{'gene_id': 'pgaptmp_000358', 'chrom': 'tig00000001_polypolish', 'tss': 334824, 'strand': '+', 'motif_start': 334514, 'rel_position_motif': -310, 'mismatches': 1, 'matched_seq': 'CGTACCACAGTTCG'}
{'gene_id': 'pgaptmp_000491', 'chrom': 'tig00000001_polypolish', 'tss': 484616, 'strand': '+', 'motif_start': 484307, 'rel_position_motif': -309, 'mismatches': 1, 'matched_seq': 'CGAAAACGCGTTCG'}
{'gene_id': 'pgaptmp_000620', 'chrom': 'ti